# Chapter 7 · 프로모션 누수

**연구질문:** 프로모션 관련 변수의 시점 누수가 예측 성능을 어떻게 왜곡하는가?

> 위에서 아래로 실행하면 끝까지 동작합니다.

## 1. 핵심 개념·수식

**누수(leakage):** 예측 시점 $t$에 알 수 없는 정보가 특성·전처리·분할에 들어가 성능이 과대평가되는 문제. 대표 경로 세 가지.
1. **전처리 누수:** 표준화·결측대체·모수 추정을 전체 자료로 한다(학습구간 통계만 써야 함).
2. **미래 정보 누수:** 중심 이동평균 등 $y_{t+k}$를 포함하는 특성.
3. **분할 누수:** 시계열에 무작위 K-겹 교차검증을 쓴다(인접 관측이 학습·검증에 섞임).

**Walk-forward(롤링 오리진) 평가:** 시점 $T_1<T_2<\dots$마다 $[1,T_k]$로 학습하고 $(T_k,T_k+h]$를 평가한다. 폴드별 오차의 분포(평균·분산·최악)를 본다.

**Diebold–Mariano 검정:** 손실차 $d_t=L(e^{A}_t)-L(e^{B}_t)$의 평균이 0인지 검정한다. $\mathrm{DM}=\bar d/\sqrt{\widehat{\mathrm{LRV}}(d)/n}$.

**부트스트랩 신뢰구간:** 폴드 손실차를 블록 부트스트랩하여 우열의 불확실성을 본다.

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
def aggregate_online_retail(df):
    # UCI Online Retail II: 취소 송장(번호가 C로 시작)과 음수 수량을 제외한 일별 매출
    df = df.copy(); df.columns = [str(c).strip() for c in df.columns]
    inv = df["Invoice"].astype(str)
    df = df[~inv.str.startswith("C") & (df["Quantity"] > 0) & (df["Price"] > 0)]
    rev = (df["Quantity"] * df["Price"]).groupby(pd.to_datetime(df["InvoiceDate"]).dt.normalize()).sum()
    s = rev.asfreq("D", fill_value=0.0)
    if len(s) < 200:
        raise ValueError("관측 부족")
    return s

def synthetic_sales(n=730, seed=SEED):
    rng = np.random.default_rng(seed); t = np.arange(n); dow = (t % 7)
    promo = np.zeros(n); 
    for s0 in range(40, n - 7, 60): promo[s0:s0 + 5] = 1.0
    mu = 1000 + 0.6 * t + 150 * np.sin(2 * np.pi * t / 365.25) - 200 * (dow >= 5) + 350 * promo
    return pd.DataFrame({"sales": np.maximum(mu + rng.normal(0, 60, n), 0), "promo": promo},
                        index=pd.date_range("2023-01-01", periods=n, freq="D"))

def load_business(force_synth=False):
    s, kind = None, "SYNTHETIC (NOT real data): 요일·연간 계절성과 프로모션이 있는 일별 매출"
    if USE_LIVE_DATA and not force_synth and ensure("openpyxl"):
        try:
            import zipfile
            z = zipfile.ZipFile(io.BytesIO(fetch_bytes("https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip", timeout=180)))
            name = [n for n in z.namelist() if n.lower().endswith(".xlsx")][0]
            sheets = pd.read_excel(io.BytesIO(z.read(name)), sheet_name=None)
            s = aggregate_online_retail(pd.concat(sheets.values(), ignore_index=True))
            kind = "REAL: UCI Online Retail II 일별 매출 (CC BY 4.0, 취소 송장 제외)"
        except Exception as e:
            print("Online Retail II 수집 실패 -> 합성 대체:", type(e).__name__, str(e)[:120])
    if s is not None:
        return dict(y=s.rename("sales"), m=7, freq="D", label="일별 매출", unit="GBP", kind=kind)
    df = synthetic_sales()
    return dict(y=df["sales"], df=df, m=7, freq="D", label="일별 매출(합성)", unit="원", kind=kind)

D = load_business(True)
y, DATA_KIND, M, FREQ = D["y"], D["kind"], D["m"], D["freq"]
print("DATA_KIND:", DATA_KIND); print("관측 수:", len(y), "| 기간:", y.index.min().date(), "~", y.index.max().date())

## 2. 프로모션 누수

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
df_s = D["df"].copy() if "df" in D else None
if df_s is None:
    # 실제 자료에는 프로모션 표지가 없다 -> 누수 시연은 합성 자료로 수행한다
    ns_ = {}
    df_s = synthetic_sales(); DATA_KIND = "SYNTHETIC (NOT real data): 프로모션 표지가 있는 합성 매출 (누수 시연용)"
sales, promo = df_s["sales"].astype(float), df_s["promo"].astype(float)
dow = pd.get_dummies(sales.index.dayofweek, prefix="dow", dtype=float); dow.index = sales.index
def X_base():
    return pd.concat([pd.DataFrame({f"lag_{k}": sales.shift(k) for k in (1, 7, 14)}), dow, promo.rename("promo_today")], axis=1)
def evaluate(X, label, shuffle=False):
    d = pd.concat([X, sales.rename("t")], axis=1).dropna(); n_tr = int(len(d) * 0.75)
    idx = np.random.default_rng(SEED).permutation(len(d)) if shuffle else np.arange(len(d))
    tr, te = idx[:n_tr], idx[n_tr:]
    m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.06, random_state=SEED).fit(d.drop(columns="t").iloc[tr], d["t"].iloc[tr])
    return {"설정": label, "RMSE": rmse(d["t"].iloc[te], m.predict(d.drop(columns="t").iloc[te]))}
rows = [evaluate(X_base(), "정상: 오늘 프로모션(사전 계획) + 과거 판매")]
Xl = X_base(); Xl["promo_effect_proxy"] = sales.shift(-1) - sales               # 누수: 내일 판매와의 차이(미래 정보)
rows.append(evaluate(Xl, "누수1: 미래 판매가 들어간 파생변수"))
Xl2 = X_base(); Xl2["promo_next7_mean_sales"] = sales.shift(-1).rolling(7).mean()   # 누수: 프로모션 이후 판매 평균(사후 결과)
rows.append(evaluate(Xl2, "누수2: 프로모션 이후 판매 평균(사후 변수)"))
rows.append(evaluate(X_base(), "누수3: 무작위 분할", shuffle=True))
show(pd.DataFrame(rows).set_index("설정"))
print("RMSE가 정상 설정보다 크게 낮은 설정은 누수다. 프로모션 변수는 '예측 시점에 알려진 계획'일 때만 합법적 특성이다.")

## 3. 프로모션 정보의 실제 가치

In [ ]:
# 프로모션 정보가 실제로 도움이 되는지(정직한 평가): 시간순 walk-forward, 프로모션 포함/제외 비교
def wf(X, n_folds=6, fold=60):
    d = pd.concat([X, sales.rename("t")], axis=1).dropna(); n = len(d); start = n - n_folds * fold; out = []
    for k in range(n_folds):
        a, b = start + k * fold, start + (k + 1) * fold
        m = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.06, random_state=SEED).fit(d.drop(columns="t").iloc[:a], d["t"].iloc[:a])
        out.append(rmse(d["t"].iloc[a:b], m.predict(d.drop(columns="t").iloc[a:b])))
    return np.array(out)
w_with, w_wo = wf(X_base()), wf(X_base().drop(columns="promo_today"))
print("Walk-forward RMSE | 프로모션 포함:", w_with.round(1), "평균", round(w_with.mean(), 1))
print("Walk-forward RMSE | 프로모션 제외:", w_wo.round(1), "평균", round(w_wo.mean(), 1))

## 3. 결과 해석 작성
1. 정상 설정과 누수 설정의 성능 차이를 표로 제시하고, **어떤 누수 경로가 가장 큰 왜곡을 만들었는지** 설명한다.
2. 폴드(또는 지역·기간) 간 변동을 보고 "평균은 좋지만 최악의 경우가 나쁘다"를 구분한다.
3. 결론(모형 우열, 효과 방향)이 설정 변경에도 유지되는가? 유지되지 않으면 "강건하다"가 아니라 "민감하다"로 쓴다.
4. 한계: 폴드 수, 이벤트 수, 합성자료의 한계, 위약검정이 모든 식별 가정을 증명하지는 않는다는 점.

**제출:** 정상/누수 비교표, 폴드별 결과, 강건성 표, 한계 단락. 누수 점검 목록(`validation/Leakage_Checklist.md`)과 대조하라.